In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
import numpy as np
from collections import Counter
from scipy import sparse
import anndata as ad
import pickle
import matplotlib.pyplot as plt
from torch import nn
import torch

In [ ]:
adata_64um = sc.read_h5ad('../Visium/HumanKidney_Visium_HD_64um_full.h5ad')

In [ ]:
adata_64um

In [ ]:
adata_64um.uns['spatialdata_attrs']

In [ ]:
with open(f'./VisiumHD_HumanKidney_spatialDE_genes.pickle', 'rb') as f:
    gene_list = pickle.load(f)
f.close()

In [ ]:
len(gene_list)

In [ ]:
import sys
import os

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("../SuperFocus")

# Add the directory to sys.path
sys.path.append(script_dir)

In [ ]:
from importlib import reload
import src.model_training.model_training_orig
import src.model_training.processing_image_features
reload(src.model_training.model_training_orig)
reload(src.model_training.processing_image_features)
from src.model_training.model_training_orig import training_orig

In [ ]:
sample_id = 'Visium_HD_HumanKidney'
he_id = 'Visium_HD_Human_Kidney_FFPE_tissue_image_tiled_only'

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training_orig(adata=adata_64um, pt_model_dir=f'../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=116.8, feature_list=gene_list, 
              save_model_dir=f'./logs_{sample_id}_reproduce', hidden_dims=[2048, 2048, 1024, 1024], activations=None, residual=False, norm=False, out_act=nn.ReLU(), 
              include_tanhshrink=False, sample_id=sample_id, training_mode='Full', selected_model='lite', kfold=5, cell_embed_dir='./VisiumHD_HumanKidney_cellvit_embed.pickle')

In [ ]:
import src.post_process
from importlib import reload
reload(src.post_process)
from src.post_process import post_training_processing

In [ ]:
adata = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_reproduce/checkpoints/model.best.pth', hidden_dims=[2048, 2048, 1024, 1024], 
                                               activations=None, out_act=nn.ReLU(), residual=False, norm=False, selected_model='lite', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level1.pickle', feature_list=gene_list, datatype='RNA', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=1, sample_id=sample_id, radius=116.8)

In [ ]:
sc.pl.spatial(adata, img_key=None, color='leiden_res_0.8')

In [ ]:
import src.model_training.model_training_refined
reload(src.model_training.model_training_refined)
from src.model_training.model_training_refined import training

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata_64um, pt_model_dir=f'../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=116.8,
         refine_factor=2, feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_reproduce/checkpoints/model.best.pth', 
         save_model_dir=f'./logs_{sample_id}_reproduce_2x', hidden_dims=[2048, 2048, 1024, 1024], activations=None, 
         residual=False, norm=False, out_act=nn.ReLU(), include_tanhshrink=False, sample_id=sample_id, selected_model='lite', 
         training_mode='Full', 
         kfold=5, cell_embed_dir='./VisiumHD_HumanKidney_cellvit_embed.pickle', step_size=1000, random_state=42)

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata_64um, pt_model_dir=f'../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=116.8,
         refine_factor=4, feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_reproduce_2x/best_model_fold5.pth', 
         save_model_dir=f'./logs_{sample_id}_reproduce_4x', hidden_dims=[2048, 2048, 1024, 1024], activations=None, 
         residual=False, norm=False, out_act=nn.ReLU(), include_tanhshrink=False, sample_id=sample_id, selected_model='lite', 
         training_mode='Full', 
         kfold=5, cell_embed_dir='./VisiumHD_HumanKidney_cellvit_embed.pickle', step_size=1000, random_state=42)

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata_64um, pt_model_dir=f'../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=116.8,
         refine_factor=8, feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_reproduce_4x/best_model_fold4.pth', 
         save_model_dir=f'./logs_{sample_id}_reproduce_8x', hidden_dims=[2048, 2048, 1024, 1024], activations=None, 
         residual=False, norm=False, out_act=nn.ReLU(), include_tanhshrink=False, sample_id=sample_id, selected_model='lite', 
         training_mode='Full', 
         kfold=5, cell_embed_dir='./VisiumHD_HumanKidney_cellvit_embed.pickle', step_size=50, gamma=0.5, random_state=42)

### apply it to the VisiumHD ROI

In [ ]:
refine_factor=8

with open(f'./{sample_id}_full_data_level{refine_factor}.pickle', 'rb') as file:
    cellvit_marker_array, other_marker_array_sub, gene_exp = pickle.load(file)

In [ ]:
with open(f'./{sample_id}_scaler_refine_level{refine_factor}.pickle', 'rb') as file:
    CODEX_HIST_scaler = pickle.load(file)

In [ ]:
x_train_full = CODEX_HIST_scaler.transform(cellvit_marker_array)

In [ ]:
other_marker_array_sub

In [ ]:
radius = 116.8 / refine_factor
train_coordx = np.array(other_marker_array_sub.iloc[:, other_marker_array_sub.columns.get_loc('lower_x_refined')] + radius)
train_coordy = np.array(other_marker_array_sub.iloc[:, other_marker_array_sub.columns.get_loc('lower_y_refined')] + radius)

In [ ]:
with open(f'./{sample_id}_full_training_data_level{refine_factor}.pickle', 'wb') as file:
    pickle.dump((cellvit_marker_array, gene_exp, train_coordx, train_coordy, x_train_full, other_marker_array_sub), file)

In [ ]:
refine_factor=8
adata_8x = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_reproduce_8x/best_model_fold3.pth', hidden_dims=[2048, 2048, 1024, 1024], 
                                               activations=None, out_act=nn.ReLU(), residual=False, norm=False, selected_model='lite', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_full_training_data_level{refine_factor}.pickle', feature_list=gene_list, datatype='RNA', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=refine_factor, sample_id=sample_id, radius=116.8)

In [ ]:
adata_8x

In [ ]:
adata_8x.layers['counts']

In [ ]:
sc.pl.spatial(adata_8x, img_key=None, color='leiden_res_0.6', size=1.5)

### apply it to the whole tissue

In [ ]:
from src.model_training.processing_image_features import whole_image_all_cells_features
from src.model_training import MLP

In [ ]:
refine_factor = 8

In [ ]:
cell_markers, cell_counts, cell_coords = whole_image_all_cells_features(pt_model_dir=f'../CellViT/Visium/preprocessing/{he_id}/{he_id}/cell_detection', radius=116.8/refine_factor, cell_embed_dir='./VisiumHD_HumanKidney_cellvit_embed.pickle')

In [ ]:
cellvit_marker_array = np.hstack((cell_markers, cell_counts))

In [ ]:
with open(f'./{sample_id}_scaler_refine_level{refine_factor}.pickle', 'rb') as file:
    CODEX_HIST_scaler = pickle.load(file)

In [ ]:
with open(f'./{sample_id}_gene_exp_pd.pickle', 'rb') as file:
    dbit_pd = pickle.load(file)

In [ ]:
codex_cell_marker_scaled = CODEX_HIST_scaler.transform(cellvit_marker_array)

In [ ]:
hidden_dims=[2048, 2048, 1024, 1024]
activations=None
residual=False
norm=False
out_act=nn.ReLU()
model = MLP.Model_lite(n_inp=3609, n_out=2010, hidden_dims=hidden_dims, activations=activations, out_activation=out_act)

In [ ]:
state_dict = torch.load(f'./logs_{sample_id}_reproduce_8x/best_model_fold3.pth', map_location=torch.device('cpu')) # Load on CPU
model.load_state_dict(state_dict)

In [ ]:
model.eval()

with torch.no_grad():
    output_full = model(torch.tensor(codex_cell_marker_scaled).to(torch.float32))

In [ ]:
predicted_gene_full= np.array(output_full)
#predicted_gene_full = np.maximum(0, predicted_gene_full)

## Create adata object
adata = ad.AnnData(predicted_gene_full)#ad.AnnData(sparse.csc_matrix(predicted_gene))

#adata.obs_names = list(dbit_pd.index)
adata.var_names = list(dbit_pd.columns)

adata.obs['x'] = np.array(cell_coords['pxl_col_in_fullres'])
adata.obs['y'] = np.array(cell_coords['pxl_row_in_fullres'])
adata.obs['section_id'] = ['VisiumHD']*adata.n_obs
adata.layers['counts'] = adata.X.copy()
#adata.raw = adata

#sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# regress out unwanted variables, if normalized to total number of cell, no need to include this step
#adata.layers["scaled"] = adata.X
#sc.pp.regress_out(adata, ['total_counts'])

# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs = 50)

In [ ]:
#adata.write_h5ad('./DBiT_imputation-CellViT-VisiumHD_HumanKidney-8x-spatialDE-all_cells.h5ad')